In [1]:
from IPython.display import Javascript, display

# 定义保活脚本：每 5 分钟 (300000 毫秒) 执行一次
keep_alive_js = """
setInterval(function() {
    // 在控制台打印日志，保持前端活跃
    console.log('[MyBinder Keep-Alive] Ping at ' + new Date().toLocaleTimeString());
    
    // 模拟一次极轻微的鼠标移动事件，防止浏览器或 Jupyter 判定为空闲
    const event = new MouseEvent('mousemove', {
        view: window,
        bubbles: true,
        cancelable: true
    });
    document.dispatchEvent(event);
}, 300000); // 300000 毫秒 = 5 分钟

console.log('[MyBinder Keep-Alive] 保活脚本已启动！');
"""

# 执行注入
display(Javascript(keep_alive_js))
print("✅ 前端保活脚本已注入！只要此浏览器标签页不关闭，MyBinder 就不会因空闲断开。")

<IPython.core.display.Javascript object>

✅ 前端保活脚本已注入！只要此浏览器标签页不关闭，MyBinder 就不会因空闲断开。


In [2]:
import time
import threading
from IPython.display import clear_output, display

def keep_alive_kernel(interval_minutes=5):
    """后台保活线程"""
    counter = 0
    while True:
        time.sleep(interval_minutes * 60)
        counter += 1
        # 打印极短的信息，保持 WebSocket 活跃，\r 确保不会刷屏
        print(f"\r[Kernel Keep-Alive] 活跃中... 已运行 {counter * interval_minutes} 分钟", end="")
        
# 启动守护线程（主程序结束时它会自动退出）
threading.Thread(target=keep_alive_kernel, args=(5,), daemon=True).start()
print("✅ Python 后台保活线程已启动！")

✅ Python 后台保活线程已启动！


In [3]:
import time
import threading
import subprocess
import os
from datetime import datetime

def auto_save_and_push(interval_minutes=10):
    """定时保存并推送代码"""
    while True:
        time.sleep(interval_minutes * 60)
        try:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
            print(f"\n[Auto-Save] 开始自动保存和同步 ({timestamp})...")
            
            # 1. 添加所有更改
            subprocess.run(["git", "add", "."], capture_output=True, text=True)
            # 2. 提交更改
            subprocess.run(["git", "commit", "-m", f"Auto-save from MyBinder: {timestamp}"], capture_output=True, text=True)
            # 3. 推送到远程仓库
            result = subprocess.run(["git", "push"], capture_output=True, text=True)
            
            if result.returncode == 0:
                print("✅ [Auto-Save] 同步成功！")
            else:
                print(f"❌ [Auto-Save] 推送失败: {result.stderr}")
        except Exception as e:
            print(f"❌ [Auto-Save] 发生错误: {e}")

# 启动后台自动保存线程
threading.Thread(target=auto_save_and_push, args=(10,), daemon=True).start()
print("✅ 自动保存与 Git 同步线程已启动（每 10 分钟执行一次）！")

✅ 自动保存与 Git 同步线程已启动（每 10 分钟执行一次）！
